# UCS420: Cognitive Computing
## Assignment 4 – A Cognitive FAQ System Using Pandas (Nova 2.0)

**Name:** Mehakdeep Kaur  
**Roll No:** 1024170364  
**Group:** 3Q32  
**Assignment:** 4

---

### What are we doing in this assignment?

In this assignment, we build a small **Cognitive FAQ System using Pandas**. The system stores frequently asked questions (FAQs) in a Pandas DataFrame and then uses simple keyword matching to find the most relevant answer to a user's query.

The assignment teaches us how to:

1. Create a personalized knowledge base using our roll number.
2. Store FAQ questions, answers, keywords, and categories in a Pandas DataFrame.
3. Score a user's query by comparing its words with FAQ keywords.
4. Retrieve questions belonging to a particular category.
5. Add a new keyword to an FAQ and save the updated data as a CSV file.
6. Count FAQs in each category using `groupby()`.
7. Handle **ties**, so that all equally good matches are displayed instead of selecting only one.

The assignment contains **6 questions (Q1–Q6)**. The code below is written so that it can be copied and run directly in **Anaconda Jupyter Notebook**.

## Q1. Build Your Personalized Knowledge Base

### Question
Take the college roll number, extract its digits, and build a Pandas DataFrame with exactly 6 FAQ entries:
- 4 fixed entries given in the assignment.
- 2 personalized entries created from the **last two digits** of the roll number.

For each last digit `d`, calculate:

`category = ["billing", "account", "general"][d % 3]`

### Our roll number
`1024170364`

The last two digits are **6** and **4**.

- Digit **6** → `6 % 3 = 0` → **billing**
- Digit **4** → `4 % 3 = 1` → **account**

Therefore, we create:
- One personalized **billing** FAQ.
- One personalized **account** FAQ.

### What are we doing?
We are creating a small knowledge base. Each row represents one FAQ and contains a question, its answer, keywords, and category.

In [1]:
import pandas as pd

# Student details
roll_no = "1024170364"

# Last two digits
last_two_digits = [int(d) for d in roll_no[-2:]]

# Category mapping from the assignment
categories = ["billing", "account", "general"]

print("Roll No:", roll_no)
print("Last two digits:", last_two_digits)

for d in last_two_digits:
    print(f"Digit {d} -> {d} % 3 = {d % 3} -> {categories[d % 3]}")

# Four fixed entries from the question
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

# Two personalized entries based on roll number digits 6 and 4
personalized_entries = [
    {
        "question": "how can i check my billing statement",
        "answer": "You can check your billing statement in the billing section of your account.",
        "keywords": "billing statement bill charges",
        "category": categories[last_two_digits[0] % 3]
    },
    {
        "question": "how do i update my registered mobile number",
        "answer": "Go to Account Settings and update your registered mobile number.",
        "keywords": "account mobile number update",
        "category": categories[last_two_digits[1] % 3]
    }
]

# Combine all 6 entries
faq_entries = fixed_entries + personalized_entries

# Create DataFrame
df = pd.DataFrame(faq_entries)

print("\nFinal 6-row FAQ DataFrame:")
display(df)

Roll No: 1024170364
Last two digits: [6, 4]
Digit 6 -> 6 % 3 = 0 -> billing
Digit 4 -> 4 % 3 = 1 -> account

Final 6-row FAQ DataFrame:


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how can i check my billing statement,You can check your billing statement in the bi...,billing statement bill charges,billing
5,how do i update my registered mobile number,Go to Account Settings and update your registe...,account mobile number update,account


## Q2. Generate and Score a Hypothesis

### Question
Implement a scoring function that takes a query string and returns all matching entries ranked by confidence.

### What are we doing?
Here, the user enters a question such as:

`How do I pay my fee?`

The program:
1. Converts the query into lowercase words.
2. Compares those words with the keywords of every FAQ.
3. Counts how many query words match the FAQ keywords.
4. Uses the number of matches as the **confidence score**.
5. Sorts the matching FAQs from highest score to lowest score.

A higher score means that the FAQ has more matching keywords and is therefore considered more relevant.

In [2]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        keyword_words = set(row["keywords"].lower().split())
        matches = query_words.intersection(keyword_words)
        score = len(matches)

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "matched_keywords": ", ".join(sorted(matches)),
                "confidence_score": score
            })

    result_df = pd.DataFrame(results)

    if result_df.empty:
        return result_df

    return result_df.sort_values(
        by="confidence_score",
        ascending=False
    ).reset_index(drop=True)


# Example query
query = "how can i pay the fee"
print("Query:", query)
display(score_query(query, df))

Query: how can i pay the fee


,question,answer,category,matched_keywords,confidence_score
0,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,"fee, pay",2
1,what is the annual fee,The annual fee is Rs 500.,billing,fee,1


## Q3. Find FAQs from the Same Category

### Question
Write a function `same_category(category_name, df)` that returns all questions belonging to a given category. Call it using the category of one personalized entry.

### What are we doing?
We are filtering the DataFrame using the `category` column.

One of our personalized entries is created from digit **6**, which maps to the **billing** category. Therefore, we use `billing` to demonstrate the function.

In [3]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()]


# Category of the first personalized entry
personalized_category = personalized_entries[0]["category"]

print("Personalized entry category:", personalized_category)
print("\nQuestions in the same category:")
display(same_category(personalized_category, df))

Personalized entry category: billing

Questions in the same category:


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how can i check my billing statement,You can check your billing statement in the bi...,billing statement bill charges,billing


## Q4. Add a New Keyword and Save the DataFrame

### Question
Pick any one entry, ask the user to input a new keyword, add it to that entry's keywords, and save the entire updated DataFrame to:

`1024170364_faq_data.csv`

### What are we doing?
We are allowing the knowledge base to be updated by the user.

The code below:
1. Selects the first FAQ entry.
2. Takes a new keyword using `input()`.
3. Adds the keyword to the existing keyword list.
4. Saves the complete DataFrame as a CSV file.

When running this cell in Jupyter, type a keyword such as **charges** when prompted.

In [4]:
# Select the first FAQ entry
entry_index = 0

print("Selected FAQ:", df.loc[entry_index, "question"])
new_keyword = input("Enter a new keyword to add: ").strip().lower()

if new_keyword:
    existing_keywords = df.loc[entry_index, "keywords"].split()
    if new_keyword not in existing_keywords:
        existing_keywords.append(new_keyword)
        df.loc[entry_index, "keywords"] = " ".join(existing_keywords)
        print("Keyword added successfully.")
    else:
        print("This keyword already exists.")
else:
    print("No keyword entered.")

# Save the complete updated DataFrame
csv_filename = f"{roll_no}_faq_data.csv"
df.to_csv(csv_filename, index=False)

print("Updated DataFrame:")
display(df)
print(f"CSV file saved as: {csv_filename}")

Selected FAQ: what is the annual fee


Enter a new keyword to add:  charges


Keyword added successfully.
Updated DataFrame:


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge charges,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how can i check my billing statement,You can check your billing statement in the bi...,billing statement bill charges,billing
5,how do i update my registered mobile number,Go to Account Settings and update your registe...,account mobile number update,account


CSV file saved as: 1024170364_faq_data.csv


## Q5. Count FAQ Entries Per Category

### Question
Using `groupby`, print how many FAQ entries you have per category.

### What are we doing?
We are grouping the DataFrame by the **category** column and counting the number of FAQs in each group.

This helps us understand how our knowledge base is distributed among billing, account, and general questions.

In [5]:
category_counts = df.groupby("category").size()

print("Number of FAQ entries per category:")
print(category_counts)

Number of FAQ entries per category:
category
account    2
billing    3
general    1
dtype: int64


## Q6. Handle Tied Highest Scores

### Question
Modify the Q2 scoring function so that if two or more entries tie for the highest score, it prints **all matching entries** instead of silently choosing one.

Demonstrate:
1. One query that produces a tie.
2. One query that does not produce a tie.

### What are we doing?
A normal search system might return only one answer even when multiple FAQs have exactly the same confidence score. That can hide useful information.

The improved function:
- Calculates scores for all FAQs.
- Finds the highest score.
- Displays **every FAQ with that highest score**.
- Also works normally when there is only one highest-scoring FAQ.

The query **`fee`** demonstrates a tie because it appears in both fixed billing FAQs:
- `what is the annual fee`
- `how can i pay the fee`

In [6]:
def score_query_with_ties(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        keyword_words = set(row["keywords"].lower().split())
        matches = query_words.intersection(keyword_words)
        score = len(matches)

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "matched_keywords": ", ".join(sorted(matches)),
                "confidence_score": score
            })

    result_df = pd.DataFrame(results)

    if result_df.empty:
        print("No matching FAQ found.")
        return result_df

    result_df = result_df.sort_values(
        by="confidence_score",
        ascending=False
    ).reset_index(drop=True)

    highest_score = result_df["confidence_score"].max()
    top_matches = result_df[result_df["confidence_score"] == highest_score]

    print(f"Highest confidence score: {highest_score}")
    print(f"Number of top matches: {len(top_matches)}")

    if len(top_matches) > 1:
        print("Tie detected. All highest-scoring entries are shown below:")
    else:
        print("No tie. The highest-scoring entry is shown below:")

    display(top_matches)
    return top_matches


# Demonstration 1: Query that produces a tie
print("DEMONSTRATION 1: TIE QUERY")
score_query_with_ties("fee", df)

# Demonstration 2: Query that does not produce a tie
print("\nDEMONSTRATION 2: NON-TIE QUERY")
score_query_with_ties("password reset", df)

DEMONSTRATION 1: TIE QUERY
Highest confidence score: 1
Number of top matches: 2
Tie detected. All highest-scoring entries are shown below:


,question,answer,category,matched_keywords,confidence_score
0,what is the annual fee,The annual fee is Rs 500.,billing,fee,1
1,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,fee,1



DEMONSTRATION 2: NON-TIE QUERY
Highest confidence score: 2
Number of top matches: 1
No tie. The highest-scoring entry is shown below:


,question,answer,category,matched_keywords,confidence_score
0,how to reset password,Go to Settings > Reset Password.,account,"password, reset",2


,question,answer,category,matched_keywords,confidence_score
0,how to reset password,Go to Settings > Reset Password.,account,"password, reset",2
